In [1]:
from pathlib import Path
from datetime import datetime
import re

import pandas as pd
import pdfplumber


CURRENT = Path.cwd() #Find the folder where the notebook is currently running.

if CURRENT.name == "notebooks":
    PROJECT_ROOT = CURRENT.parent
else:
    PROJECT_ROOT = CURRENT


PDF_FOLDER = PROJECT_ROOT / "data" / "raw" / "daily"
OUTPUT_FOLDER = PROJECT_ROOT / "data" / "processed"

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


PDF_NAME = "IDSP-Daily-Report-01.09.2026.pdf"

PDF_PATH = PDF_FOLDER / PDF_NAME

if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_NAME}"
    )

print("Using:", PDF_PATH.name)

Using: IDSP-Daily-Report-01.09.2026.pdf


In [2]:
def clean(value):
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def to_count(value):
    text = clean(value).replace(",", "")

    if text == "-":
        return 0

    if text.isdigit():
        return int(text)

    raise ValueError(
        f"Invalid number found: {value!r}"
    )

In [3]:
SETTINGS = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "intersection_tolerance": 5,
    "snap_tolerance": 3,
    "join_tolerance": 3,
}


with pdfplumber.open(PDF_PATH) as pdf:
    if len(pdf.pages) < 2:
        raise ValueError("PDF does not have page 2.")

    tables = pdf.pages[1].extract_tables(SETTINGS)


if not tables:
    raise ValueError("No table found on page 2.")


table = max(tables, key=len)

print("Rows:", len(table))
print("Columns:", max(len(row) for row in table))

Rows: 40
Columns: 31


In [4]:
header_text = " ".join(
    clean(cell)
    for row in table[:3]
    for cell in row
)


date_match = re.search(
    r"\b\d{2}-\d{2}-\d{2}\b",
    header_text
)


if not date_match:
    raise ValueError("Report date not found.")


report_date = datetime.strptime(
    date_match.group(),
    "%d-%m-%y"
).date().isoformat()


print("Report date:", report_date)

Report date: 2026-09-01


In [5]:
DISTRICTS = {
    "TVM": "Thiruvananthapuram",
    "KLM": "Kollam",
    "PTA": "Pathanamthitta",
    "IDK": "Idukki",
    "KTM": "Kottayam",
    "ALP": "Alappuzha",
    "EKM": "Ernakulam",
    "TSR": "Thrissur",
    "PKD": "Palakkad",
    "MPM": "Malappuram",
    "KKD": "Kozhikode",
    "WYD": "Wayanad",
    "KNR": "Kannur",
    "KSD": "Kasaragod",
}

COLUMNS = [
    "row_number",
    "district_code",

    "fever_op",
    "fever_ip",

    "chikungunya_suspected",
    "chikungunya_confirmed",
    "chikungunya_deaths",

    "dengue_suspected",
    "dengue_confirmed",
    "dengue_deaths",

    "leptospirosis_suspected",
    "leptospirosis_confirmed",
    "leptospirosis_deaths",

    "add_confirmed",
    "chickenpox_confirmed",
    "hepatitis_a_confirmed",

    "cholera_suspected_cases",
    "cholera_suspected_deaths",
    "cholera_confirmed_cases",
    "cholera_confirmed_deaths",

    "aes_confirmed",
    "je_confirmed",

    "malaria_pv_imported",
    "malaria_pv_indigenous",
    "malaria_pf_imported",
    "malaria_pf_indigenous",
    "malaria_mixed_imported",
    "malaria_mixed_indigenous",
    "malaria_deaths",

    "scrub_typhus_confirmed",
    "influenza_confirmed",
]

print("Defined columns:", len(COLUMNS))

Defined columns: 31


In [6]:
expected_column_count = len(COLUMNS)

actual_column_count = max(
    len(row) for row in table
)


if actual_column_count != expected_column_count:
    raise ValueError(
        "New or changed PDF layout detected.\n"
        f"Expected columns: {expected_column_count}\n"
        f"Detected columns: {actual_column_count}\n"
        "A new extraction schema is required."
    )


print(
    "PASS: PDF has the expected",
    expected_column_count,
    "columns."
)

PASS: PDF has the expected 31 columns.


In [7]:
expected_headers = {
    2: "FEVER",
    4: "CG",
    7: "DENGUE",
    10: "LEPTO",
    16: "CHOLERA",
    22: "MALARIA",
}


header_row = table[2]

header_errors = []


for position, expected_name in expected_headers.items():

    actual_name = clean(
        header_row[position]
    ).upper()

    if expected_name not in actual_name:

        header_errors.append({
            "position": position,
            "expected": expected_name,
            "detected": actual_name,
        })


if header_errors:
    raise ValueError(
        "Disease columns changed:\n"
        f"{header_errors}"
    )


print("PASS: Important disease headers are in the expected positions.")

PASS: Important disease headers are in the expected positions.


In [9]:
valid_codes = set(DISTRICTS)
valid_codes.add("TOT")


# Select district rows
rows = []

for row in table:

    if len(row) != len(COLUMNS):
        continue

    district_code = clean(row[1]).upper()

    if district_code in valid_codes:
        rows.append(row)


# Create the dataframe
df = pd.DataFrame(
    rows,
    columns=COLUMNS
)


# Clean district codes before validating them
df["district_code"] = (
    df["district_code"]
    .map(clean)
    .str.upper()
)


# Check the total number of rows
if len(df) != 15:
    raise ValueError(
        f"Expected 15 rows, found {len(df)}"
    )


# Check the 14 district codes
found_districts = set(
    df.loc[
        df["district_code"] != "TOT",
        "district_code"
    ]
)

expected_districts = set(DISTRICTS)


if found_districts != expected_districts:

    missing = expected_districts - found_districts
    unexpected = found_districts - expected_districts

    raise ValueError(
        f"Missing districts: {missing}\n"
        f"Unexpected districts: {unexpected}"
    )


# Check duplicate district codes
if df["district_code"].duplicated().any():

    duplicates = df.loc[
        df["district_code"].duplicated(),
        "district_code"
    ].tolist()

    raise ValueError(
        f"Duplicate district codes: {duplicates}"
    )


# Convert disease counts to integers
number_columns = COLUMNS[2:]

for column in number_columns:
    df[column] = df[column].map(to_count)


print("PASS: 14 unique districts and one TOT row found.")
print("Rows extracted:", len(df))

df[["district_code", "dengue_confirmed"]]

PASS: 14 unique districts and one TOT row found.
Rows extracted: 15


,district_code,dengue_confirmed
0,TVM,20
1,KLM,11
2,PTA,3
3,IDK,1
4,KTM,0
5,ALP,0
6,EKM,7
7,TSR,22
8,PKD,3
9,MPM,10


In [10]:
total_rows = df[
    df["district_code"] == "TOT"
]


if len(total_rows) != 1:
    raise ValueError(
        f"Expected one TOT row, found {len(total_rows)}"
    )


published_total = total_rows.iloc[0]


district_df = df[
    df["district_code"] != "TOT"
].copy()


validation_results = []


for column in number_columns:

    calculated_total = int(
        district_df[column].sum()
    )

    expected_total = int(
        published_total[column]
    )

    validation_results.append({
        "column": column,
        "calculated_total": calculated_total,
        "published_total": expected_total,
        "match": calculated_total == expected_total,
    })


validation_df = pd.DataFrame(
    validation_results
)


failed_checks = validation_df[
    validation_df["match"] == False
]


if not failed_checks.empty:
    raise AssertionError(
        "Total validation failed:\n"
        + failed_checks.to_string(index=False)
    )


print(
    f"PASS: {len(validation_df)}/"
    f"{len(validation_df)} totals matched."
)

validation_df

PASS: 29/29 totals matched.


,column,calculated_total,published_total,match
0,fever_op,12480,12480,True
1,fever_ip,94,94,True
2,chikungunya_suspected,0,0,True
3,chikungunya_confirmed,1,1,True
4,chikungunya_deaths,0,0,True
5,dengue_suspected,174,174,True
6,dengue_confirmed,79,79,True
7,dengue_deaths,1,1,True
8,leptospirosis_suspected,11,11,True
9,leptospirosis_confirmed,18,18,True


In [11]:
# Remove the PDF serial-number column
district_df = district_df.drop(
    columns="row_number"
)


# Add the complete district name
district_df.insert(
    1,
    "district_name",
    district_df["district_code"].map(DISTRICTS)
)


# Add report information
district_df.insert(
    0,
    "report_date",
    report_date
)

district_df.insert(
    1,
    "period_type",
    "daily"
)

district_df.insert(
    2,
    "source_filename",
    PDF_PATH.name
)

district_df.insert(
    3,
    "schema_version",
    "daily_v1"
)


# Create output paths
district_output = (
    OUTPUT_FOLDER
    / f"district_table_{report_date}.csv"
)

validation_output = (
    OUTPUT_FOLDER
    / f"district_validation_{report_date}.csv"
)


# Save the results
district_df.to_csv(
    district_output,
    index=False
)

validation_df.to_csv(
    validation_output,
    index=False
)


print("District data saved:", district_output)
print("Validation saved:", validation_output)


# Show Kannur
district_df[
    district_df["district_code"] == "KNR"
]

District data saved: C:\Users\vinee\Downloads\rag_chatbot_kerala\data\processed\district_table_2026-09-01.csv
Validation saved: C:\Users\vinee\Downloads\rag_chatbot_kerala\data\processed\district_validation_2026-09-01.csv


,report_date,period_type,source_filename,schema_version,district_code,district_name,fever_op,fever_ip,chikungunya_suspected,chikungunya_confirmed,...,je_confirmed,malaria_pv_imported,malaria_pv_indigenous,malaria_pf_imported,malaria_pf_indigenous,malaria_mixed_imported,malaria_mixed_indigenous,malaria_deaths,scrub_typhus_confirmed,influenza_confirmed
12,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,KNR,Kannur,795,6,0,0,...,0,0,0,0,0,0,0,0,0,2
